# One-Year Cross-Model Comparison on Inferred Triplets

Comparativa en el **mismo anio calendario** para productos proxy entre Amazon/Favorita/M5.

Modelos:
- TSB_static (multi-step clasico)
- TSB_WF (walk-forward, 1 paso con actualizacion diaria)
- SARIMAX
- Hurdle
- Regime (selector Hurdle/SARIMAX)

Nota: si un triplete no tiene traslape temporal comun de 1 anio entre los 3 datasets, se excluye automaticamente.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / 'src').exists() and (cur / 'reports').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.run_cross_market_peak_comparison import (
    _load_amazon_selected,
    _load_favorita_selected,
    _load_m5_selected,
)
from src.models.tsb_model import TSBModel
from src.models.sarimax_model import SARIMAXModel
from src.models.hurdle_model import HurdleModel
from src.models.regime_forecast_engine import RegimeForecastEngine

print('Repo root:', REPO_ROOT)


In [ ]:
# Config
TARGET_YEAR = 2015
YEAR_MODE = 'target'  # 'target' or 'auto_common'
TRAIN_RATIO = 0.75
AMAZON_FILE = 'Health_and_Household.jsonl.gz'
AMAZON_MAX_ROWS = 100_000

ADD_INTERMITTENT_ARTICLE = True
INTERMITTENT_TRIPLET = {
    'inferred_product': 'Intermittent stress test (proxy)',
    'M5_WALMART': 'HOUSEHOLD_2_108_WI_1_validation',
    'FAVORITA': 'item_916886',
    'AMAZON_2023': 'B0BZTLKX7H',
}

triplets_csv = REPO_ROOT / 'reports' / 'cross_market_peak_comparison' / 'inferred_triplets.csv'
triplets = pd.read_csv(triplets_csv)

if ADD_INTERMITTENT_ARTICLE:
    extra_triplet = pd.DataFrame([INTERMITTENT_TRIPLET])
    triplets = pd.concat([triplets, extra_triplet], ignore_index=True)
    triplets = triplets.drop_duplicates(subset=['M5_WALMART', 'FAVORITA', 'AMAZON_2023'], keep='first').reset_index(drop=True)

triplets


In [ ]:
m5_ids = triplets['M5_WALMART'].astype(str).tolist()
fav_ids = triplets['FAVORITA'].astype(str).tolist()
amz_ids = triplets['AMAZON_2023'].astype(str).tolist()

series_map_by_ds = {
    'M5_WALMART': _load_m5_selected(str(REPO_ROOT / 'data' / 'raw' / 'm5'), m5_ids),
    'FAVORITA': _load_favorita_selected(str(REPO_ROOT / 'data' / 'raw' / 'favorita'), fav_ids, store_nbr=1),
    'AMAZON_2023': _load_amazon_selected(
        str(REPO_ROOT / 'data' / 'raw' / 'amazon_2023' / 'review_categories'),
        AMAZON_FILE,
        amz_ids,
        max_rows=AMAZON_MAX_ROWS,
    ),
}
{k: len(v) for k, v in series_map_by_ds.items()}


In [ ]:
def build_calendar_features(df: pd.DataFrame):
    out = df.copy().sort_values('date').reset_index(drop=True)
    out['date'] = pd.to_datetime(out['date'])
    out['price'] = 0.0
    if 'price' in out.columns:
        out['price'] = out['price'].ffill().bfill().fillna(0.0)

    out['dow'] = out['date'].dt.dayofweek
    out['month'] = out['date'].dt.month
    out['is_weekend'] = (out['dow'] >= 5).astype(int)

    feature_cols = ['price', 'dow', 'month', 'is_weekend']
    out[feature_cols] = out[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out, feature_cols

def filter_year(df: pd.DataFrame, year: int):
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'])
    out = out[(out['date'] >= f'{year}-01-01') & (out['date'] <= f'{year}-12-31')]
    return out.sort_values('date').reset_index(drop=True)

def split_by_ratio(df: pd.DataFrame, train_ratio: float = 0.75):
    n = len(df)
    split = int(n * train_ratio)
    split = min(max(split, 30), n - 30)
    train = df.iloc[:split].copy()
    test = df.iloc[split:].copy()
    return train, test

def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mask = y_true > 0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.sum() > 0 else np.nan
    return mae, rmse, mape

def get_triplet_overlap_year(trip_row, series_map, min_days=330):
    starts, ends = [], []
    for ds in ['M5_WALMART', 'FAVORITA', 'AMAZON_2023']:
        sid = str(trip_row[ds])
        df = series_map[ds][sid]
        d = pd.to_datetime(df['date'])
        starts.append(d.min())
        ends.append(d.max())

    overlap_start = max(starts)
    overlap_end = min(ends)
    if overlap_end < overlap_start:
        return None, overlap_start.date(), overlap_end.date()

    candidate_years = list(range(overlap_start.year, overlap_end.year + 1))
    valid_years = []
    for y in candidate_years:
        y_start = pd.Timestamp(f'{y}-01-01')
        y_end = pd.Timestamp(f'{y}-12-31')
        start = max(overlap_start, y_start)
        end = min(overlap_end, y_end)
        days = (end - start).days + 1
        if days >= min_days:
            valid_years.append(y)

    if not valid_years:
        return None, overlap_start.date(), overlap_end.date()

    return max(valid_years), overlap_start.date(), overlap_end.date()

def tsb_walk_forward(y_train, y_test):
    # Re-fit TSB every step with observed history; forecast 1-step ahead.
    history = list(np.asarray(y_train, dtype=float))
    y_test = np.asarray(y_test, dtype=float)
    preds = np.zeros(len(y_test), dtype=float)

    for i in range(len(y_test)):
        m = TSBModel().fit(np.asarray(history, dtype=float))
        preds[i] = m.forecast(1)[0][0]
        history.append(float(y_test[i]))

    return preds

def run_models_on_series(df_year: pd.DataFrame, train_ratio: float = 0.75):
    feat_df, feature_cols = build_calendar_features(df_year)
    train_df, test_df = split_by_ratio(feat_df, train_ratio=train_ratio)

    y_train = train_df['sales'].astype(float).values
    y_test = test_df['sales'].astype(float).values
    X_train = train_df[feature_cols].astype(float).values
    X_test = test_df[feature_cols].astype(float).values

    preds = {}

    preds['TSB_static'] = TSBModel().fit(y_train).forecast(len(y_test))[0]
    preds['TSB_WF'] = tsb_walk_forward(y_train, y_test)
    preds['SARIMAX'] = SARIMAXModel().fit(y_train, X_train).forecast(len(y_test), X_test)[0]

    hurdle = HurdleModel().fit(X_train, y_train)
    preds['Hurdle'] = hurdle.forecast(X_test, X_train, y_train)[0]

    regime = RegimeForecastEngine()
    preds['Regime'] = regime.run(y_train, X_train, y_test, X_test)[0]

    return train_df, test_df, y_test, preds


In [ ]:
def demand_profile(df: pd.DataFrame, year: int):
    y = filter_year(df, year)['sales'].astype(float).values
    if len(y) == 0:
        return {'days': 0, 'zero_rate': np.nan, 'adi': np.nan}

    zero_rate = float(np.mean(y == 0))
    non_zero = float(np.sum(y > 0))
    adi = float(len(y) / non_zero) if non_zero > 0 else np.inf
    return {'days': int(len(y)), 'zero_rate': zero_rate, 'adi': adi}

if ADD_INTERMITTENT_ARTICLE:
    intermittent_rows = []
    row = triplets[triplets['inferred_product'] == INTERMITTENT_TRIPLET['inferred_product']]
    if not row.empty:
        row = row.iloc[0]
        for ds in ['AMAZON_2023', 'FAVORITA', 'M5_WALMART']:
            sid = str(row[ds])
            prof = demand_profile(series_map_by_ds[ds][sid], TARGET_YEAR)
            intermittent_rows.append({
                'dataset': ds,
                'series_id': sid,
                'days_in_target_year': prof['days'],
                'zero_rate': prof['zero_rate'],
                'adi': prof['adi'],
            })

    if intermittent_rows:
        print('Perfil de intermitencia del articulo agregado (anio target):')
        display(pd.DataFrame(intermittent_rows).sort_values('dataset').reset_index(drop=True))


In [ ]:
availability_rows = []
valid_triplets = []
excluded_triplets = []

for _, trip in triplets.iterrows():
    product = trip['inferred_product']
    auto_year, ov_start, ov_end = get_triplet_overlap_year(trip, series_map_by_ds, min_days=330)

    if YEAR_MODE == 'target':
        year_to_use = TARGET_YEAR
    else:
        year_to_use = auto_year

    all_have_year = True
    per_ds_days = {}
    for ds in ['M5_WALMART', 'FAVORITA', 'AMAZON_2023']:
        sid = str(trip[ds])
        df = series_map_by_ds[ds][sid]
        n_days_year = len(filter_year(df, year_to_use)) if year_to_use is not None else 0
        per_ds_days[ds] = n_days_year
        if n_days_year < 330:
            all_have_year = False

    availability_rows.append({
        'inferred_product': product,
        'auto_common_year': auto_year,
        'overlap_start': ov_start,
        'overlap_end': ov_end,
        'year_used': year_to_use,
        'days_M5_WALMART': per_ds_days['M5_WALMART'],
        'days_FAVORITA': per_ds_days['FAVORITA'],
        'days_AMAZON_2023': per_ds_days['AMAZON_2023'],
        'is_valid_triplet_for_year': all_have_year,
    })

    if all_have_year:
        valid_triplets.append((trip, year_to_use))
    else:
        excluded_triplets.append((product, year_to_use, per_ds_days))

availability_df = pd.DataFrame(availability_rows)
display(availability_df)

if excluded_triplets:
    print('Tripletes excluidos por falta de traslape de 1 anio en el year seleccionado:')
    for product, year_used, days_map in excluded_triplets:
        print(f' - {product} | year={year_used} | days={days_map}')


In [ ]:
rows = []
plot_store = {}

for trip, eval_year in valid_triplets:
    product = trip['inferred_product']
    for ds in ['AMAZON_2023', 'FAVORITA', 'M5_WALMART']:
        sid = str(trip[ds])
        df = series_map_by_ds[ds][sid]
        df_year = filter_year(df, eval_year)

        train_df, test_df, y_test, preds = run_models_on_series(df_year, train_ratio=TRAIN_RATIO)
        plot_store[(product, ds)] = {'test_df': test_df, 'y_test': y_test, 'preds': preds, 'year': eval_year}

        for model_name, y_pred in preds.items():
            mae, rmse, mape = calc_metrics(y_test, y_pred)
            rows.append({
                'year': eval_year,
                'inferred_product': product,
                'dataset': ds,
                'series_id': sid,
                'model': model_name,
                'train_days': len(train_df),
                'test_days': len(test_df),
                'mae': mae,
                'rmse': rmse,
                'mape': mape,
            })

results = pd.DataFrame(rows).sort_values(['inferred_product', 'dataset', 'model']).reset_index(drop=True)
results


In [ ]:
print('MAE pivot (menor es mejor)')
display(results.pivot_table(index=['inferred_product', 'dataset'], columns='model', values='mae').round(4))
print('RMSE pivot (menor es mejor)')
display(results.pivot_table(index=['inferred_product', 'dataset'], columns='model', values='rmse').round(4))
print('MAPE pivot (menor es mejor)')
display(results.pivot_table(index=['inferred_product', 'dataset'], columns='model', values='mape').round(2))


In [ ]:
model_colors = {
    'TSB_static': '#9467bd',
    'TSB_WF': '#e377c2',
    'SARIMAX': '#ff7f0e',
    'Hurdle': '#2ca02c',
    'Regime': '#1f77b4',
}

valid_products = sorted(results['inferred_product'].unique().tolist())
datasets = ['AMAZON_2023', 'FAVORITA', 'M5_WALMART']

fig, axes = plt.subplots(nrows=len(valid_products), ncols=len(datasets), figsize=(18, 4 * max(len(valid_products), 1)), sharex=False)
if len(valid_products) == 1:
    axes = np.array([axes])

for i, product in enumerate(valid_products):
    for j, ds in enumerate(datasets):
        ax = axes[i, j]
        payload = plot_store[(product, ds)]
        test_df = payload['test_df']
        y_test = payload['y_test']
        preds = payload['preds']
        year_used = payload['year']

        ax.plot(test_df['date'], y_test, color='black', lw=1.6, label='Actual')
        for model_name, y_pred in preds.items():
            ax.plot(test_df['date'], y_pred, color=model_colors[model_name], lw=1.2, label=model_name)

        ax.set_title(f'{product} | {ds} | year={year_used}', fontsize=9)
        ax.tick_params(axis='x', rotation=35)
        ax.grid(alpha=0.2)

        if i == 0 and j == 0:
            ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
out_dir = REPO_ROOT / 'reports' / 'one_year_triplet_model_comparison'
out_dir.mkdir(parents=True, exist_ok=True)

availability_path = out_dir / 'triplet_year_availability.csv'
availability_df.to_csv(availability_path, index=False)

if len(results) > 0:
    mode_tag = f'{YEAR_MODE}_{TARGET_YEAR}' if YEAR_MODE == 'target' else YEAR_MODE
    out_csv = out_dir / f'model_comparison_triplets_{mode_tag}.csv'
    results.to_csv(out_csv, index=False)
    print('Saved:', availability_path)
    print('Saved:', out_csv)
else:
    print('Saved:', availability_path)
    print('No valid triplets for requested year mode.')

results
